## 5.6. GPU

In [15]:
!nvidia-smi

Sun Jan 12 19:39:23 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 566.03                 Driver Version: 566.03         CUDA Version: 12.7     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce GTX 1650      WDDM  |   00000000:01:00.0  On |                  N/A |
| 30%   35C    P8              8W /   75W |    1052MiB /   4096MiB |     26%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

### 5.6.1. Computation Device

In [16]:
import torch
from torch import nn

torch.device('cpu'), torch.device('cuda'), torch.device('cuda:1')

(device(type='cpu'), device(type='cuda'), device(type='cuda', index=1))

In [17]:
torch.cuda.device_count()

1

In [18]:
def try_gpu(i=0): #@save
  """if exist, return gpu(i), else return cpu()"""
  if torch.cuda.device_count() >= i + 1:
    return torch.device(f'cuda:{i}')
  return torch.device('cpu')

def try_all_gpus(): #@save
  """return all accessible GPU, if no, return [cpu(),]"""
  devices = [ torch.device(f'cuda:{i}')
              for i in range(torch.cuda.device_count())]
  return devices if devices else [torch.device('cpu')]

try_gpu(), try_gpu(10), try_all_gpus()

(device(type='cuda', index=0),
 device(type='cpu'),
 [device(type='cuda', index=0)])

### 5.6.2 Tensor and GPU

In [19]:
x = torch.tensor([1, 2, 3])
x.device

device(type='cpu')

#### 5.6.2.1 Save to GPU

In [20]:
X = torch.ones(2, 3, device=try_gpu())
X

tensor([[1., 1., 1.],
        [1., 1., 1.]], device='cuda:0')

In [21]:
Y = torch.rand(2, 3, device=try_gpu(1))
Y


tensor([[0.9279, 0.4630, 0.4203],
        [0.0778, 0.8402, 0.3687]])

#### 5.6.2.2 Copy

In [22]:
Z = X.cpu()
print(X)
print(Z)

tensor([[1., 1., 1.],
        [1., 1., 1.]], device='cuda:0')
tensor([[1., 1., 1.],
        [1., 1., 1.]])


In [23]:
Y + Z

tensor([[1.9279, 1.4630, 1.4203],
        [1.0778, 1.8402, 1.3687]])

In [27]:
Z.cpu() is Z

True

#### 5.6.2.3 Side Notes

### 5.6.3 Neural Network and GPU

In [29]:
net = nn.Sequential(nn.Linear(3, 1))
net = net.to(device=try_gpu())

In [30]:
net(X)

tensor([[-0.7753],
        [-0.7753]], device='cuda:0', grad_fn=<AddmmBackward0>)

In [31]:
net[0].weight.data.device

device(type='cuda', index=0)